In [3]:
import requests, zipfile
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

ZENODO_RECORDS = {
    "eparchos": 4095301,
    #"stavronikita_53": 5595669,
    #"stavronikita_79": 5578136,
    #"stavronikita_114": 5578251,
    #"zenon_papyri": 6565706,
    #"meleagre_model": 10932742,
}

def download_zenodo(name, record_id, out_dir=DATA_DIR, unzip=True):
    target = out_dir / name
    target.mkdir(parents=True, exist_ok=True)
    meta = requests.get(f"https://zenodo.org/api/records/{record_id}", timeout=30)
    meta.raise_for_status()
    for f in meta.json()["files"]:
        path = target / f["key"]
        if path.exists():
            print(f"skip {path} (already there)")
            continue
        print(f"downloading {f['key']} ({f['size'] / 1e6:.1f} MB)")
        with requests.get(f["links"]["self"], stream=True, timeout=60) as r:
            r.raise_for_status()
            with open(path, "wb") as out:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    out.write(chunk)
        if unzip and path.suffix == ".zip":
            with zipfile.ZipFile(path) as z:
                z.extractall(target)
    return target

for name, rid in ZENODO_RECORDS.items():
    download_zenodo(name, rid)

skip data/eparchos/eparchos.zip (already there)
